<a href="https://colab.research.google.com/github/vencov/FAV_upsidedown/blob/main/RatioPitch/EX_octave_vs_mel_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Octave Steps vs. Mel-Scale Steps: A Listening Demonstration

This notebook plays two ascending tone sequences starting from the same reference frequency:

1. **Octave steps** — each tone is exactly **double the frequency (Hz)** of the previous one.
2. **Mel-scale steps** — each tone is **double the *perceived pitch*** of the previous one,
   according to the mel scale (Stevens, Volkmann & Newman, 1937). Since the mel scale is
   compressive, doubling perceived pitch requires a *smaller* frequency ratio than 2:1,
   especially at higher frequencies.

Listen to both sequences and notice: the octave sequence keeps sounding like a series of
equal musical jumps, while the mel sequence's steps get perceptually more similar in size
even though the Hz gaps are shrinking relative to an octave.


In [4]:
import numpy as np
from ipywidgets import Dropdown, Button, Output, VBox, HBox, Label
from IPython.display import Audio, display, clear_output

SAMPLE_RATE = 44100

def pure_tone(freq_hz, duration_s=0.8, amp=0.3, sr=SAMPLE_RATE):
    t = np.linspace(0, duration_s, int(sr * duration_s), endpoint=False)
    fade = int(0.01 * sr)
    envelope = np.ones_like(t)
    envelope[:fade] = np.linspace(0, 1, fade)
    envelope[-fade:] = np.linspace(1, 0, fade)
    return amp * envelope * np.sin(2 * np.pi * freq_hz * t)

def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

def mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)

def silence(duration_s=0.3, sr=SAMPLE_RATE):
    return np.zeros(int(sr * duration_s))

def concat_sequence(freqs, gap_s=0.3):
    """Glue tones together with a short silent gap so they play back-to-back in order."""
    parts = []
    for f in freqs:
        parts.append(pure_tone(f))
        parts.append(silence(gap_s))
    return np.concatenate(parts)

def build_octave_sequence(start_hz, max_hz=8000):
    seq = [start_hz]
    current = start_hz
    while True:
        nxt = current * 2
        if nxt > max_hz:
            break
        seq.append(nxt)
        current = nxt
    return seq

def build_mel_sequence(start_hz, max_hz=8000):
    seq = [start_hz]
    current_mel = hz_to_mel(start_hz)
    while True:
        current_mel *= 2
        nxt = mel_to_hz(current_mel)
        if nxt > max_hz:
            break
        seq.append(nxt)
    return seq


## 1. Octave sequence

Each tone below is exactly **2× the frequency** of the previous one — this is what a
sequence of octaves sounds like.


In [5]:
start_dropdown = Dropdown(options=[125, 250, 500, 1000], value=250, description='Start (Hz):')
octave_btn = Button(description='Play octave sequence', button_style='info')
octave_out = Output()

def play_octaves(_):
    with octave_out:
        clear_output(wait=True)
        freqs = build_octave_sequence(start_dropdown.value)
        print("Frequencies (Hz):", [f"{f:.0f}" for f in freqs])
        print("Mel values:      ", [f"{hz_to_mel(f):.0f}" for f in freqs])
        display(Audio(concat_sequence(freqs), rate=SAMPLE_RATE, autoplay=True, normalize=False))

octave_btn.on_click(play_octaves)
display(VBox([start_dropdown, octave_btn, octave_out]))


## 2. Mel-scale sequence

Each tone below is chosen so that its **perceived pitch (mel) is exactly double** the
previous tone's — not its frequency. Use the *same* starting frequency as above for a fair
comparison.


In [6]:
mel_btn = Button(description='Play mel-scale sequence', button_style='warning')
mel_out = Output()

def play_mel(_):
    with mel_out:
        clear_output(wait=True)
        freqs = build_mel_sequence(start_dropdown.value)
        print("Frequencies (Hz):", [f"{f:.0f}" for f in freqs])
        print("Mel values:      ", [f"{hz_to_mel(f):.0f}" for f in freqs])
        display(Audio(concat_sequence(freqs), rate=SAMPLE_RATE, autoplay=True, normalize=False))

mel_btn.on_click(play_mel)
display(VBox([mel_btn, mel_out]))


## 3. Visualize the comparison

The plot below shows the standard mel curve (frequency → perceived pitch), with your
octave sequence and your mel sequence marked on top of it. Notice how the octave points
are evenly spaced along the frequency (x) axis but crowd together on the mel (y) axis at
higher frequencies — while the mel-sequence points are evenly spaced in mel but spread out
increasingly along frequency.


In [7]:
import matplotlib.pyplot as plt

plot_btn = Button(description='Show comparison graph', button_style='success')
plot_out = Output()

def show_comparison(_):
    with plot_out:
        clear_output(wait=True)
        start = start_dropdown.value
        octave_freqs = build_octave_sequence(start)
        mel_freqs = build_mel_sequence(start)

        f_curve = np.linspace(start * 0.8, 8000, 500)
        mel_curve = hz_to_mel(f_curve)

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(f_curve, mel_curve, color='lightgray', label='Standard mel curve', zorder=1)

        ax.plot(octave_freqs, hz_to_mel(np.array(octave_freqs)), 'o-', color='C0',
                label='Octave sequence (x2 Hz each step)', zorder=3)
        for i, f in enumerate(octave_freqs):
            ax.annotate(str(i), (f, hz_to_mel(f)), textcoords="offset points",
                        xytext=(0, 8), fontsize=8, color='C0', ha='center')

        ax.plot(mel_freqs, hz_to_mel(np.array(mel_freqs)), 's-', color='C1',
                label='Mel sequence (x2 mel each step)', zorder=3)
        for i, f in enumerate(mel_freqs):
            ax.annotate(str(i), (f, hz_to_mel(f)), textcoords="offset points",
                        xytext=(0, -14), fontsize=8, color='C1', ha='center')

        ax.set_xscale('log')
        ax.set_xlabel('Frequency (Hz, log scale)')
        ax.set_ylabel('Perceived pitch (mel)')
        ax.set_title(f'Octave steps vs. mel steps, starting from {start} Hz')
        ax.legend()
        ax.grid(alpha=0.3, which='both')
        plt.show()

plot_btn.on_click(show_comparison)
display(VBox([plot_btn, plot_out]))
